# Phase 1 chunk 6 Explorations

## Original code below. With Beginner Exploration already done

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time


# 1.  Configuration
# for reproducibility
torch.manual_seed(42)

# Hyperparameters
LEARNING_RATE  = 0.001
BATCH_SIZE = 64
EPOCHS = 15

# Set device
device = "cuda" if torch.cuda.is_available() else 'cpu'
print(f"Using the device : {device}")

# 2. Loading the FashionMNIST dataset
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5),(0.5))
    ]
)

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True,
                                      transform=transform)

test_dataset =  datasets.FashionMNIST(root='./data',
                                      train=False,
                                      download=True,
                                      transform=transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
    )

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Define the model
class FashionMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.net = nn.Sequential(
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.net(self.flatten(x))

model = FashionMNIST().to(device)

# 4. Loss and Optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = LEARNING_RATE)

# 5. The Training and Validation Functions

def train_one_epoch(loader, model, loss_fn, optimizer):
    model.train() # Set model for training mode
    running_loss = 0.0
    for batch, (X, y) in enumerate(loader):
        X, y = X.to(device), y.to(device)
        
        # 1. Forward pass
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    
    return running_loss / len(loader)


def validate_one_epoch(loader, model, loss_fn):
    model.eval() # set model to evaluation mode
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred = model(X)

            running_loss += loss_fn(pred, y).item()


            # calculate accuracy
            total += y.size(0)
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    avg_loss = running_loss / len(loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


# --- 6. The Main Training Loop ---
print("Starting training...")
start_time = time.time()

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(train_loader, model, loss_fn, optimizer)
    val_loss, val_acc = validate_one_epoch(test_loader, model, loss_fn)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

end_time = time.time()
print(f"Finished training in {end_time - start_time:.2f} seconds.")



Using the device : cuda
Starting training...
Epoch 1/15 | Train Loss: 0.5123 | Val Loss: 0.4196 | Val Acc: 84.53%
Epoch 2/15 | Train Loss: 0.3784 | Val Loss: 0.3967 | Val Acc: 85.44%
Epoch 3/15 | Train Loss: 0.3406 | Val Loss: 0.4100 | Val Acc: 85.44%
Epoch 4/15 | Train Loss: 0.3177 | Val Loss: 0.3583 | Val Acc: 86.93%
Epoch 5/15 | Train Loss: 0.2983 | Val Loss: 0.3326 | Val Acc: 88.29%
Epoch 6/15 | Train Loss: 0.2821 | Val Loss: 0.3358 | Val Acc: 88.20%
Epoch 7/15 | Train Loss: 0.2711 | Val Loss: 0.3346 | Val Acc: 88.03%
Epoch 8/15 | Train Loss: 0.2578 | Val Loss: 0.3570 | Val Acc: 86.95%
Epoch 9/15 | Train Loss: 0.2469 | Val Loss: 0.3356 | Val Acc: 88.31%
Epoch 10/15 | Train Loss: 0.2375 | Val Loss: 0.3608 | Val Acc: 87.57%
Epoch 11/15 | Train Loss: 0.2279 | Val Loss: 0.3610 | Val Acc: 88.00%
Epoch 12/15 | Train Loss: 0.2200 | Val Loss: 0.3358 | Val Acc: 88.73%
Epoch 13/15 | Train Loss: 0.2138 | Val Loss: 0.3397 | Val Acc: 88.34%
Epoch 14/15 | Train Loss: 0.2035 | Val Loss: 0.3493 | 

## Intermediate

In [3]:
# Add code to save your model weights.

torch.save(model.state_dict(), "fashion_mnist_model.pth")

# Why save weights and not the entire model?
 
 Saving only the model weights (state_dict) is more flexible and efficient. It allows you to:
 1. **Load weights into a different model architecture**: You can load the weights into a model with a different architecture, as long as the layer names match.
 2. **Reduce file size**: The state_dict only contains the parameters, not the entire model structure, making it smaller and faster to save/load.
 3. **Easier experimentation**: You can easily switch between different model architectures or configurations by loading different state_dicts.


# Advanced

### Early stopping

In [6]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time

# Creating an Earlystoping class
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        """
        Args:
            patience (int) : How many epochs to wait adter last improvement before stopping.
            min_delta (float) : Minimum change to qualify as an improvement.
        """
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0.0
        self.best_loss = np.inf
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0 # Reset patience counter

        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True   


# 1.  Configuration
# for reproducibility
torch.manual_seed(42)

# Hyperparameters
LEARNING_RATE  = 0.001
BATCH_SIZE = 64
EPOCHS = 15

# Set device
device = "cuda" if torch.cuda.is_available() else 'cpu'
print(f"Using the device : {device}")

# 2. Loading the FashionMNIST dataset
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5),(0.5))
    ]
)

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True,
                                      transform=transform)

test_dataset =  datasets.FashionMNIST(root='./data',
                                      train=False,
                                      download=True,
                                      transform=transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
    )

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Define the model
class FashionMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.net = nn.Sequential(
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.net(x)

model = FashionMNIST().to(device)

# 4. Loss and Optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = LEARNING_RATE)

# 5. The Training and Validation Functions

def train_one_epoch(loader, model, loss_fn, optimizer):
    model.train() # Set model for training mode
    running_loss = 0.0
    for batch, (X, y) in enumerate(loader):
        X, y = X.to(device), y.to(device)
        
        # 1. Forward pass
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    
    return running_loss / len(loader)


def validate_one_epoch(loader, model, loss_fn):
    model.eval() # set model to evaluation mode
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred = model(X)

            running_loss += loss_fn(pred, y).item()


            # calculate accuracy
            total += y.size(0)
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    avg_loss = running_loss / len(loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

early_stopping = EarlyStopping(patience=3, min_delta=0.001)
# --- 6. The Main Training Loop ---
print("Starting training...")
start_time = time.time()

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(train_loader, model, loss_fn, optimizer)
    val_loss, val_acc = validate_one_epoch(test_loader, model, loss_fn)
    early_stopping(val_loss)
    if early_stopping.early_stop:
        break
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

end_time = time.time()
print(f"Finished training in {end_time - start_time:.2f} seconds.")



Using the device : cuda
Starting training...
Epoch 1/15 | Train Loss: 0.5123 | Val Loss: 0.4196 | Val Acc: 84.53%
Epoch 2/15 | Train Loss: 0.3784 | Val Loss: 0.3967 | Val Acc: 85.44%
Epoch 3/15 | Train Loss: 0.3406 | Val Loss: 0.4100 | Val Acc: 85.44%
Epoch 4/15 | Train Loss: 0.3177 | Val Loss: 0.3583 | Val Acc: 86.93%
Epoch 5/15 | Train Loss: 0.2983 | Val Loss: 0.3326 | Val Acc: 88.29%
Epoch 6/15 | Train Loss: 0.2821 | Val Loss: 0.3358 | Val Acc: 88.20%
Epoch 7/15 | Train Loss: 0.2711 | Val Loss: 0.3346 | Val Acc: 88.03%
Finished training in 247.81 seconds.
